# utils\nGenerated from 00_common/utils.py for Databricks notebook import.\n

In [ ]:
"""Common utility helpers for Databricks ingestion framework."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport logging\nimport sys\nimport uuid\nfrom datetime import datetime, timezone\nfrom typing import Any, Dict\n\n\ndef utc_now_iso() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef new_load_id() -> str:\n    return str(uuid.uuid4())\n\n\ndef row_hash(payload: Dict[str, Any]) -> str:\n    canonical = json.dumps(payload, sort_keys=True, default=str)\n    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()\n\n\ndef require_keys(payload: Dict[str, Any], required: list[str]) -> None:\n    missing = [k for k in required if k not in payload]\n    if missing:\n        raise ValueError(f"Missing required keys: {missing}")\n\n\ndef truncate_str(value: str, max_chars: int = 500) -> str:\n    """Truncate a string to *max_chars*, replacing newlines for single-line output."""\n    msg = str(value).replace("\n", " ").strip()\n    return msg[:max_chars]\n\n\nclass _JsonFormatter(logging.Formatter):\n    """Emit each log record as a single-line JSON object."""\n\n    def format(self, record: logging.LogRecord) -> str:\n        payload: Dict[str, Any] = {\n            "ts": datetime.fromtimestamp(record.created, tz=timezone.utc).isoformat(),\n            "level": record.levelname,\n            "logger": record.name,\n            "message": record.getMessage(),\n        }\n        if record.exc_info:\n            payload["exc_info"] = self.formatException(record.exc_info)\n        return json.dumps(payload, default=str)\n\n\ndef get_logger(name: str = "ingestion_framework") -> logging.Logger:\n    """Return a logger that writes structured JSON lines to stdout.\n\n    Safe to call multiple times with the same name — returns the existing\n    logger without adding duplicate handlers.\n    """\n    logger = logging.getLogger(name)\n    if logger.handlers:\n        return logger\n    logger.setLevel(logging.DEBUG)\n    handler = logging.StreamHandler(sys.stdout)\n    handler.setFormatter(_JsonFormatter())\n    logger.addHandler(handler)\n    logger.propagate = False\n    return logger\n